# AGENT WEB · Colab SDXL
Google AI Proの契約アカウントで、ランタイムをL4 GPUに設定してください。
このノートブックはSDXLをColabのGPUで実行します。Gemini APIや有料APIは呼びません。
接続にはCloudflareの一時HTTPSトンネルを使用します。認証された画像生成機能だけを公開し、agent-web本体は公開しません。
接続URLとキーはagent-webのColab画面へ入力してください。キーは公開・共有しないでください。
生成画像は最終的にVMに保存されます。Colab内のファイルはランタイム削除で失われます。

参照画像対応版です。SDXLに加えてIP-Adapterと画像エンコーダーを読み込みます。


In [ ]:
%pip install -q -U diffusers transformers accelerate safetensors
%pip uninstall -y torchao


インストール後、**ランタイム → セッションを再起動**してください。その後は以下のセルから実行します。

In [ ]:
import torch
from diffusers import DiffusionPipeline
from transformers import CLIPVisionModelWithProjection
assert torch.cuda.is_available(), "GPUを選択してください"
ADAPTER_REVISION = "018e402774aeeddd60609b4ecdb7e298259dc729"
BASE_REVISION = "462165984030d82259a11f4367a4eed129e94a7b"
image_encoder = CLIPVisionModelWithProjection.from_pretrained(
    "h94/IP-Adapter", subfolder="models/image_encoder", revision=ADAPTER_REVISION,
    torch_dtype=torch.float16,
)
pipe = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0", revision=BASE_REVISION,
    image_encoder=image_encoder,
    torch_dtype=torch.float16, variant="fp16", use_safetensors=True,
).to("cuda")
pipe.load_ip_adapter(
    "h94/IP-Adapter", subfolder="sdxl_models",
    weight_name="ip-adapter-plus_sdxl_vit-h.safetensors", revision=ADAPTER_REVISION,
    image_encoder_folder=None,
)
pipe.set_ip_adapter_scale(0.5)
print("参照画像対応SDXLの準備完了:", torch.cuda.get_device_name(0))


次のセルは接続サービスと一時トンネルを起動します。Colabを起動し直したらURLとキーも登録し直してください。

In [ ]:
"""Run in Colab after loading `pipe`. Only authenticated image jobs are exposed."""
import base64
import hmac
import hashlib
import re
import io
import json
import secrets
import threading
import uuid
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer


def start_worker(pipe=None, port=8765, reference_ready=False, motion_pipe=None):
    import torch
    token = secrets.token_urlsafe(32)
    jobs = {}
    assets = {}
    clips = {}
    asset_ready = reference_ready or motion_pipe is not None
    instance_id = str(uuid.uuid4())
    lock = threading.Lock()

    def generate(job_id, params):
        try:
            def progress(pipeline, step, timestep, kwargs):
                with lock:
                    jobs[job_id]['progress'] = round((step + 1) / params['steps'] * 100)
                return kwargs
            if params.get('mode') == 'motion':
                from PIL import Image
                reference=Image.open(io.BytesIO(assets[params['reference_asset']])).convert('RGB')
                poses=[Image.open(io.BytesIO(assets[h])).convert('RGB') for h in params['pose_assets']]
                motion_pipe.set_ip_adapter_scale(params['reference_scale'])
                frames=motion_pipe(prompt=params['prompt'], negative_prompt=params.get('negative_prompt',''),
                    width=params['width'],height=params['height'],num_frames=len(poses),
                    num_inference_steps=params['steps'],guidance_scale=params.get('guidance_scale',7.5),
                    controlnet_conditioning_scale=params.get('pose_scale',1.0),
                    conditioning_frames=poses,ip_adapter_image=reference,
                    generator=torch.Generator('cpu').manual_seed(params['seed']),
                    callback_on_step_end=progress,decode_chunk_size=1).frames[0]
                if len(frames)!=len(poses):raise ValueError('Unexpected motion frame count')
                encoded=[]
                for frame in frames:
                    out=io.BytesIO();frame.save(out,format='PNG');encoded.append(out.getvalue())
                with lock:
                    clips[job_id]=encoded
                    jobs[job_id].update(status='completed',progress=100,frame_count=len(encoded))
                return
            extra = {}
            if reference_ready:
                from PIL import Image
                extra['ip_adapter_image'] = Image.new('RGB',(224,224),(128,128,128))
                pipe.set_ip_adapter_scale(params.get('reference_scale',0) if params.get('reference_asset') else 0)
            if params.get('reference_asset'):
                from PIL import Image
                extra['ip_adapter_image'] = Image.open(io.BytesIO(assets[params['reference_asset']])).convert('RGB')
            image = pipe(prompt=params['prompt'], negative_prompt=params.get('negative_prompt', ''),
                         width=params['width'], height=params['height'],
                         num_inference_steps=params['steps'],
                         generator=torch.Generator('cuda').manual_seed(params['seed']),
                         callback_on_step_end=progress, **extra).images[0]
            output = io.BytesIO()
            image.save(output, format='PNG')
            with lock:
                jobs[job_id].update(status='completed', progress=100,
                    image=base64.b64encode(output.getvalue()).decode())
        except Exception as exc:
            with lock:
                jobs[job_id].update(status='failed', error=str(exc)[:2000])

    class Handler(BaseHTTPRequestHandler):
        def log_message(self, *args):
            pass

        def respond(self, status, body):
            raw = json.dumps(body).encode()
            self.send_response(status)
            self.send_header('Content-Type', 'application/json')
            self.send_header('Content-Length', str(len(raw)))
            self.end_headers()
            self.wfile.write(raw)

        def authorized(self):
            if not hmac.compare_digest(self.headers.get('Authorization', ''), 'Bearer ' + token):
                self.respond(401, {'error': 'Unauthorized'})
                return False
            return True

        def do_GET(self):
            if not self.authorized(): return
            if self.path == '/status':
                with lock:
                    busy = any(j['status'] == 'running' for j in jobs.values())
                self.respond(200, {'gpu': torch.cuda.get_device_name(0),
                    'model': 'AnimateDiff OpenPose' if motion_pipe is not None else 'SDXL', 'busy': busy, 'protocol': 1, 'instance_id':instance_id,
                    'capabilities':{'reference_image':reference_ready, 'text_to_image':pipe is not None, 'motion':motion_pipe is not None},
                    'adapter_revision':'018e402774aeeddd60609b4ecdb7e298259dc729' if reference_ready else None,
                    'base_revision':'462165984030d82259a11f4367a4eed129e94a7b' if reference_ready else None})
            elif re.fullmatch(r'/jobs/[0-9a-f-]+/frames/[0-9]+',self.path):
                parts=self.path.split('/');job_id=parts[2];index=int(parts[4])
                with lock:
                    clip=clips.get(job_id,[])
                    raw=clip[index] if 0<=index<len(clip) else None
                if raw is None:return self.respond(404,{'error':'Frame not available'})
                self.send_response(200);self.send_header('Content-Type','image/png');self.send_header('Content-Length',str(len(raw)));self.end_headers();self.wfile.write(raw)
            elif self.path.startswith('/jobs/'):
                with lock:
                    job = dict(jobs.get(self.path.split('/')[-1], {}))
                self.respond(200 if job else 404, job or {'error': 'Unknown job'})
            else:
                self.respond(404, {'error': 'Not found'})

        def do_POST(self):
            if not self.authorized(): return
            if self.path.startswith('/assets/'):
                try:
                    if not asset_ready: raise ValueError('Update the Colab notebook for references')
                    digest = self.path.split('/')[-1]
                    if not re.fullmatch('[a-f0-9]{64}',digest):raise ValueError('Invalid asset ID')
                    length=int(self.headers.get('Content-Length',0))
                    if not 0<length<=20*1024*1024:raise ValueError('Invalid asset size')
                    raw=self.rfile.read(length)
                    if hashlib.sha256(raw).hexdigest()!=digest:raise ValueError('Asset checksum mismatch')
                    from PIL import Image
                    with Image.open(io.BytesIO(raw)) as image:
                        if image.format!='PNG' or image.width>1024 or image.height>1024 or getattr(image,'n_frames',1)!=1:raise ValueError('Invalid reference image')
                        image.verify()
                    with lock:
                        if digest not in assets and len(assets)>=32:
                            protected={j.get('reference_asset') for j in jobs.values() if j['status']=='running'}
                            for job in jobs.values():
                                if job['status']=='running':protected.update(job.get('pose_assets',[]))
                            for old in list(assets):
                                if old not in protected:del assets[old];break
                        assets[digest]=raw
                    return self.respond(200,{'sha256':digest})
                except (ValueError,OSError):return self.respond(400,{'error':'Invalid reference asset'})
            if self.path != '/jobs': return self.respond(404, {'error': 'Not found'})
            try:
                length = int(self.headers.get('Content-Length', 0))
                if not 0 < length <= 32768: raise ValueError('Invalid request size')
                params = json.loads(self.rfile.read(length))
                job_id = str(uuid.UUID(params['id']))
                if not isinstance(params['prompt'], str) or not 0 < len(params['prompt']) <= 4000:
                    raise ValueError('Invalid prompt')
                if not isinstance(params.get('negative_prompt', ''), str) or len(params.get('negative_prompt', '')) > 4000:
                    raise ValueError('Invalid negative prompt')
                for k in ('width', 'height'):
                    if type(params[k]) is not int or not 512 <= params[k] <= 1536 or params[k] % 64:
                        raise ValueError('Invalid dimensions')
                if type(params['steps']) is not int or not 1 <= params['steps'] <= 50:
                    raise ValueError('Invalid steps')
                if type(params['seed']) is not int or not 0 <= params['seed'] < 2**32:
                    raise ValueError('Invalid seed')
                motion=params.get('mode')=='motion'
                if motion:
                    poses=params.get('pose_assets')
                    if motion_pipe is None:raise ValueError('Motion notebook required')
                    if not isinstance(poses,list) or len(poses)!=16 or any(p not in assets for p in poses):raise ValueError('16 pose images required')
                    if params['width']!=512 or params['height']!=768:raise ValueError('Motion resolution must be 512x768')
                    for key,default,lo,hi in [('guidance_scale',7.5,1,12),('pose_scale',1.,0,2)]:
                        v=params.get(key,default)
                        if type(v) not in (int,float) or not lo<=v<=hi:raise ValueError('Invalid motion scale')
                    if not params.get('reference_asset'):raise ValueError('Character reference required')
                elif pipe is None:raise ValueError('Image notebook required')
                asset=params.get('reference_asset')
                if asset:
                    scale=params.get('reference_scale',0.5)
                    if not (reference_ready or motion) or asset not in assets:raise ValueError('Reference image unavailable')
                    if type(scale) not in (int,float) or not 0<=scale<=1:raise ValueError('Invalid reference scale')
                with lock:
                    if job_id in jobs: return self.respond(200, {'id': job_id})
                    if any(j['status'] == 'running' for j in jobs.values()):
                        return self.respond(409, {'error': 'GPU is busy'})
                    jobs[job_id] = {'id': job_id, 'status': 'running', 'progress': 0, 'reference_asset':asset,'pose_assets':params.get('pose_assets',[]),'mode':'motion' if motion else 'image'}
                threading.Thread(target=generate, args=(job_id, params), daemon=True).start()
                self.respond(202, {'id': job_id})
            except (ValueError, KeyError, TypeError) as exc:
                self.respond(400, {'error': str(exc)})

    server = ThreadingHTTPServer(('127.0.0.1', port), Handler)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, token

import os
import re
import subprocess
import time
import urllib.request
from pathlib import Path

if "aw_tunnel" in globals():
    aw_tunnel.terminate()
    aw_tunnel.wait(timeout=10)
if "aw_server" in globals():
    aw_server.shutdown()
    aw_server.server_close()
aw_server, aw_token = start_worker(pipe, reference_ready=True)
binary = Path("/content/cloudflared")
if not binary.exists():
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        binary,
    )
    binary.chmod(0o700)
log = open("/content/agent-web-tunnel.log", "w+")
aw_tunnel = subprocess.Popen(
    [str(binary), "tunnel", "--url", "http://127.0.0.1:8765", "--no-autoupdate"],
    stdout=log, stderr=subprocess.STDOUT,
)
aw_url = None
for _ in range(60):
    time.sleep(1)
    log.seek(0)
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log.read())
    if match:
        aw_url = match.group(0)
        break
    if aw_tunnel.poll() is not None:
        break
if not aw_url:
    raise RuntimeError("トンネルを起動できませんでした。/content/agent-web-tunnel.log を確認してください。")
print("agent-webのColab画面に登録してください")
print("接続URL:", aw_url)
print("接続キー:", aw_token)


作業終了時は次のセルで接続を止め、**ランタイム → 接続解除してランタイムを削除**でGPUを解放してください。

In [ ]:
aw_tunnel.terminate()
aw_server.shutdown()
aw_server.server_close()
print("接続を停止しました。GPUを解放するにはランタイムを削除してください。")
